# 第 02 章：结构化输出与规范 (Pydantic)

实战体验由随意自然文本抽取到强制严格检查对象的数据蜕变。

In [2]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field
from typing import List

load_dotenv()
llm = init_chat_model(
    model="deepseek-chat",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("https://api.deepseek.com")
)
print("大脑已上线。")

大脑已上线。


## 1. 结构化模具压制展示
使用纯 `BaseModel` 定制出输出属性约束：

In [3]:
class UserProfile(BaseModel):
    """提取出的用户信息实体大全"""
    name: str = Field(..., description="用户的全名")
    age: int = Field(..., description="用户的年龄")
    interests: list[str] = Field(description="用户的兴趣爱好列表", default_factory=list)

# 给原模型挂接上 Pydantic 过滤漏网
structured_llm = llm.with_structured_output(UserProfile)

query = "我叫张三，今年25岁，平时喜欢打篮球、游泳，最近还在网吧通宵打黑神话悟空。"
result = structured_llm.invoke(query)

# 取出的完全是从 json 退化出来的静态 Python 对象！
print("提取结果的类型为:", type(result))
print(f"姓名: {result.name} | 年龄: {result.age} | 星好集合: {result.interests}")

提取结果的类型为: <class '__main__.UserProfile'>
姓名: 张三 | 年龄: 25 | 星好集合: ['打篮球', '游泳', '打黑神话悟空']


## 2. 复合立体解析树
订单地址拥有自身独立的嵌套格式对象。测试模型的嵌套追查力：

In [4]:
class DeliveryAddress(BaseModel):
    province: str = Field(description="省份，无需后缀'省'")
    city: str = Field(description="城市，无需后缀'市'")
    detail: str = Field(description="详细地址")

class OrderReceipt(BaseModel):
    """订单回执结构集合"""
    customer_name: str = Field(description="客户姓名")
    shipping_address: DeliveryAddress = Field(description="收件地址")

order_llm = llm.with_structured_output(OrderReceipt)

text = "我是老李，你要安排发件到这边：四川的成都市武侯区科华北路最里面那个66号居民楼下。"
receipt = order_llm.invoke(text)

# 层层解析
print(f"客户大字表: {receipt.customer_name}")
print(f"嵌套之城市: {receipt.shipping_address.city}")
print(f"嵌套之明细: {receipt.shipping_address.detail}")

客户大字表: 老李
嵌套之城市: 成都
嵌套之明细: 武侯区科华北路最里面那个66号居民楼下
